In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet(
    "../data/processed/flight_weather.parquet"
)

In [ ]:
print(f"Shape: {df.shape}")

df.head()

In [ ]:
df.info()

In [ ]:
df.isna().mean().sort_values(ascending=False)

In [ ]:
df[
    [
        "departure_delay_minutes",
        "precipitation",
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "wind_gusts_10m",
    ]
].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

In [ ]:
df["flight_status"].value_counts(dropna=False)

In [ ]:
df.groupby("flight_status")[
    "departure_delay_minutes"
].agg(["count", "mean", "median", "min", "max"])

In [ ]:
df.nlargest(
    20,
    "departure_delay_minutes"
)[
    [
        "origin_airport",
        "destination_airport",
        "scheduled_departure",
        "actual_departure",
        "departure_delay_minutes",
        "flight_status",
    ]
]

In [ ]:
df.nsmallest(
    20,
    "departure_delay_minutes"
)[
    [
        "origin_airport",
        "destination_airport",
        "scheduled_departure",
        "actual_departure",
        "departure_delay_minutes",
        "flight_status",
    ]
]

In [ ]:
realized = df[
    df["flight_status"] == "REALIZADO"
].copy()

delay = realized["departure_delay_minutes"]

pd.Series({
    "adiantado > 1h": (delay < -60).sum(),
    "adiantado > 2h": (delay < -120).sum(),
    "atraso > 6h": (delay > 360).sum(),
    "atraso > 12h": (delay > 720).sum(),
    "atraso > 24h": (delay > 1440).sum(),
})

In [ ]:
pd.Series({
    "adiantado > 1h": (delay < -60).mean(),
    "adiantado > 2h": (delay < -120).mean(),
    "atraso > 6h": (delay > 360).mean(),
    "atraso > 12h": (delay > 720).mean(),
    "atraso > 24h": (delay > 1440).mean(),
})

In [ ]:
realized.groupby("Situação Partida")[
    "departure_delay_minutes"
].agg(
    count="count",
    median="median",
    min="min",
    max="max",
).sort_values("median")

In [ ]:
realized.loc[
    (realized["departure_delay_minutes"] < -120)
    | (realized["departure_delay_minutes"] > 720),
    [
        "departure_delay_minutes",
        "Situação Partida",
    ],
]["Situação Partida"].value_counts(dropna=False)

In [ ]:
realized_clean = realized[
    realized["departure_delay_minutes"].between(
        -120,
        720,
    )
].copy()

In [ ]:
print(f"Antes: {len(realized):,}")
print(f"Depois: {len(realized_clean):,}")
print(f"Removidos: {len(realized) - len(realized_clean):,}")

In [ ]:
realized_clean["departure_delay_minutes"].describe(
    percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]
)

In [ ]:
(realized_clean["precipitation"] > 0).mean()

In [ ]:
realized_clean["precipitation"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99, 0.999])

In [ ]:
realized_clean.loc[
    realized_clean["precipitation"] > 0,
    "precipitation"
].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
bins = [-0.01, 0, 0.5, 1, 2, 5, 10, float("inf")]
labels = [
    "0",
    "0-0.5",
    "0.5-1",
    "1-2",
    "2-5",
    "5-10",
    "10+",
]

realized_clean["precipitation_bin"] = pd.cut(
    realized_clean["precipitation"],
    bins=bins,
    labels=labels,
)

In [ ]:
delay_by_rain = (
    realized_clean
    .groupby("precipitation_bin", observed=True)
    .agg(
        flights=("departure_delay_minutes", "size"),
        mean_delay=("departure_delay_minutes", "mean"),
        median_delay=("departure_delay_minutes", "median"),
        p95_delay=("departure_delay_minutes", lambda x: x.quantile(0.95)),
    )
)

delay_by_rain

In [ ]:
delay_by_rain.reset_index().plot(
    x="precipitation_bin",
    y="mean_delay",
    kind="bar",
    legend=False,
    figsize=(10, 5),
    title="Atraso médio por intensidade de precipitação",
    xlabel="Precipitação (mm/h)",
    ylabel="Atraso médio (min)",
)

In [ ]:
top_airports = (
    realized_clean["origin_airport"]
    .value_counts()
    .head(10)
    .index
)

delay_by_airport_rain = (
    realized_clean[
        realized_clean["origin_airport"].isin(top_airports)
    ]
    .groupby(
        ["origin_airport", "precipitation_bin"],
        observed=True,
    )
    .agg(
        flights=("departure_delay_minutes", "size"),
        mean_delay=("departure_delay_minutes", "mean"),
    )
    .reset_index()
)

delay_by_airport_rain.head(20)

In [ ]:
realized_clean["hour"] = (
    realized_clean["scheduled_departure_local"].dt.hour
)

realized_clean["month"] = (
    realized_clean["scheduled_departure_local"].dt.month
)

realized_clean["day_of_week"] = (
    realized_clean["scheduled_departure_local"].dt.dayofweek
)

realized_clean["year"] = (
    realized_clean["scheduled_departure_local"].dt.year
)

In [ ]:
realized_clean.groupby("hour").agg(
    flights=("departure_delay_minutes", "size"),
    mean_delay=("departure_delay_minutes", "mean"),
    mean_precipitation=("precipitation", "mean"),
)

In [ ]:
realized_clean.groupby("month").agg(
    flights=("departure_delay_minutes", "size"),
    mean_delay=("departure_delay_minutes", "mean"),
    mean_precipitation=("precipitation", "mean"),
)